# 01 - Qualidade dos dados

**Objetivo:** entender a estrutura do Steam Games Dataset e diagnosticar problemas de qualidade (Entendimento dos Dados).

**Lê:** `data/original/*.csv` (baixa do Kaggle automaticamente se não existirem)  
**Gera:** `data/clean/dados_limpos.parquet`

In [6]:
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
downloaded = Path("../data/downloaded")
downloaded.mkdir(parents=True, exist_ok=True)
arquivos = ["games_may2024_cleaned.csv"]
# Baixa o dataset só se os CSVs ainda não estiverem em data/downloaded
if not all((downloaded / f).exists() for f in arquivos):
    import kagglehub  # pip install kagglehub
    path = kagglehub.dataset_download("artermiloff/steam-games-dataset")
    for f in arquivos:
        shutil.copy(Path(path) / f, downloaded / f)

print([p.name for p in downloaded.iterdir()])

['games_may2024_cleaned.csv']


In [7]:

pd.set_option('display.max_columns', None)

dataset = pd.read_csv(downloaded / "games_may2024_cleaned.csv", low_memory=False)

dataset.head()

display(dataset.head())

,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,https://cdn.akamai.steamstatic.com/steam/apps/...,http://counter-strike.net/,NaN,NaN,True,False,True,0,NaN,1,4111974,Includes intense violence and blood.,"['Czech', 'Danish', 'Dutch', 'English', 'Finni...","['English', 'Vietnamese']","[{'title': 'Buy Counter-Strike 2', 'descriptio...",['Valve'],['Valve'],"['Multi-player', 'Cross-Platform Multiplayer',...","['Action', 'Free to Play']",['https://cdn.akamai.steamstatic.com/steam/app...,['http://cdn.akamai.steamstatic.com/steam/apps...,0,NaN,7024836,1029779,100000000 - 200000000,29018,753,6078,352,1362469,"{'FPS': 90076, 'Shooter': 64786, 'Multiplayer'...",87,8071426,80,56072
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,https://cdn.akamai.steamstatic.com/steam/apps/...,https://www.pubg.com,https://support.playbattlegrounds.com,NaN,True,False,False,0,NaN,37,1701431,NaN,"['English', 'Korean', 'Simplified Chinese', 'F...",[],[],"['KRAFTON, Inc.']","['KRAFTON, Inc.']","['Multi-player', 'PvP', 'Online PvP', 'Stats',...","['Action', 'Adventure', 'Massively Multiplayer...",['https://cdn.akamai.steamstatic.com/steam/app...,['http://cdn.akamai.steamstatic.com/steam/apps...,0,NaN,1379580,981860,50000000 - 100000000,0,0,0,0,590582,"{'Survival': 14611, 'Shooter': 12477, 'Battle ...",58,2365012,68,19517
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,https://cdn.akamai.steamstatic.com/steam/apps/...,http://www.dota2.com/,NaN,NaN,True,True,True,90,https://www.metacritic.com/game/pc/dota-2?ftag...,0,14329,NaN,"['Bulgarian', 'Czech', 'Danish', 'Dutch', 'Eng...","['English', 'Korean', 'Simplified Chinese', 'V...","[{'title': 'Buy Dota 2', 'description': '', 's...",['Valve'],['Valve'],"['Multi-player', 'Co-op', 'Steam Trading Cards...","['Action', 'Strategy', 'Free to Play']",['https://cdn.akamai.steamstatic.com/steam/app...,['http://cdn.akamai.steamstatic.com/steam/apps...,0,NaN,1832477,406030,200000000 - 500000000,35557,1327,1163,977,668192,"{'Free to Play': 59559, 'MOBA': 19966, 'Multip...",81,2247365,71,23832
3,271590,Grand Theft Auto V,2015-04-13,17,0.00,1,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.rockstargames.com/V/,https://support.rockstargames.com,NaN,True,False,False,96,https://www.metacritic.com/game/pc/grand-theft...,77,1641404,NaN,"['English', 'French', 'Italian', 'German', 'Sp...","['English', 'Spanish - Latin America']","[{'title': 'Buy Shark Cash Cards', 'descriptio...",['Rockstar North'],['Rockstar Games'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Adventure']",['https://shared.akamai.steamstatic.com/store_...,['http://cdn.akamai.steamstatic.com/steam/apps...,0,NaN,1557234,236827,50000000 - 100000000,12898,590,6910,235,133571,"{'Open World': 31998, 'Action': 2317



Duas situações diferentes: valores **NaN** e **zeros que significam "sem dado"** (não aparecem como faltantes, mas são).

In [8]:
nan = dataset.isna().mean().mul(100).round(1)

zeros_faltantes = pd.Series({"user_score == 0" : (dataset.user_score == 0).mean() * 100,
                             "metacritic_score == 0": (dataset.metacritic_score == 0).mean() * 100,
                             "estimated_owners == 0": (dataset.estimated_owners == "0").mean() * 100,
                             "achievements == 0": (dataset.achievements == 0).mean() * 100,
                             
}).round(1)
print(f"Colunas com NaN declarados: \n{nan[nan > 0 ].sort_values(ascending=False).to_string()}")
print(f"\nZeros que são ausência de dado (%): \n{zeros_faltantes.to_string()}")
print(f"\nAppID duplicado: {dataset.AppID.duplicated().sum()}")

Colunas com NaN declarados: 
score_rank              100.0
metacritic_url           95.3
reviews                  87.7
notes                    83.0
website                  52.5
support_url              49.7
support_email            12.6
detailed_description      0.2
about_the_game            0.2
short_description         0.1

Zeros que são ausência de dado (%): 
user_score == 0          100.0
metacritic_score == 0     95.3
estimated_owners == 0      0.0
achievements == 0         47.2

AppID duplicado: 0


**Decisões:**
- `user_score` é 100% zero: descartada.
- `score_rank` é 100% NaN: descartada támbem.
- `metacritic_score`: 95% zero. Vira NaN e entra só como flag `tem_metacritic`.
- `estimated_owners`, `positive`, `negative`, `recommendations` e `average_playtime_forever` são **consequência** da popularidade (e `estimated_owners` já vem do mesmo processo de coleta). Ficam fora das variáveis preditoras para evitar vazamento; só aparecem no notebook de análise para mostrar o problema.
- `achievements == 0` pode ser "jogo sem conquistas" mesmo, então é mantido como valor real.
- Colunas de texto/URL (descrições, imagens, `notes`, `reviews`) não entram; só viram flags de presença quando fizer sentido.
- `AppID`sem duplicados.
- `release_date` sem datas inválidas. De 1997-06-30 a 2024-05-26 

In [9]:
dataset = dataset.drop(columns=["user_score"], errors="ignore")
dataset = dataset.drop(columns=["score_rank"], errors="ignore")

dataset["tem_metacritic"] = (dataset["metacritic_score"] > 0).astype(bool)
dataset["metacritic_score"] = dataset["metacritic_score"].replace(0, np.nan)

flag = ["website", "support_url", "support_email", "notes"]
descartar = ["about_the_game", "header_image", "reviews", "screenshots", "movies",
             "detailed_description", "short_description", "metacritic_url", "packages"]

flag = [c for c in flag if c in dataset.columns]
descartar = [c for c in descartar if c in dataset.columns]

for c in flag:
    dataset[f"tem_{c}"] = dataset[c].notna() & (dataset[c].astype(str).str.strip() !="")
dataset = dataset.drop(columns=flag+descartar)

#print("AppID duplicado:", dataset.AppID.duplicated().sum()) Nenhum duplicado
dataset["release_date"] = pd.to_datetime(dataset["release_date"], errors="coerce")
#print(dataset["release_date"].isna().sum(), "datas inválidas") nenhuma data invalida tb
print(dataset["release_date"].min(), "->", dataset["release_date"].max()) #amplitude das datas de publicacao



1997-06-30 00:00:00 -> 2024-05-26 00:00:00


In [10]:
clean = Path("../data/clean")
clean.mkdir(parents=True, exist_ok=True)
dataset.to_parquet(clean / "jogos_limpos.parquet", index=False)
print(dataset.shape)

(83646, 36)
